# TriageAI: Local Deployment with Ollama [Gemma 4]

This notebook demonstrates how to fine-tune the Gemma model (using QLoRA) on a maternal health triage dataset for the **Gemma 4 Good Hackathon**. Once fine-tuned, you can export the weights and load them into a local Ollama instance for offline inference in the SANA prototype.

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes datasets accelerate

## 1. Imports and Setup
Log in to HuggingFace (ensure you have accepted the Gemma terms of use).

In [ ]:
import torch
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# Note: In Kaggle, use Kaggle Secrets to store your HF_TOKEN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=hf_token)

## 2. Load the Dataset
For this example, we create a synthetic maternal triage dataset. In practice, load your specific hackathon dataset here.

In [ ]:
data = [
    {"instruction": "Assess the maternal risk for a 32 week pregnant patient with BP 150/100, severe headache, and blurred vision.", "output": "{\"risk_level\": \"HIGH\", \"summary\": \"Patient exhibits classic signs of severe preeclampsia.\", \"referral_needed\": true}"},
    {"instruction": "Assess the maternal risk for a 12 week pregnant patient with mild nausea and no other symptoms.", "output": "{\"risk_level\": \"LOW\", \"summary\": \"Patient exhibits normal first-trimester symptoms.\", \"referral_needed\": false}"}
]
dataset = Dataset.from_list(data)

def format_instruction(example):
    # Formatting as a chat interaction for Gemma
    return f"<start_of_turn>user\n{example['instruction']}<end_of_turn>\n<start_of_turn>model\n{example['output']}<end_of_turn>"

# Apply formatting
# For TRL SFTTrainer, we just need a text column
dataset = dataset.map(lambda x: {"text": format_instruction(x)})

## 3. Load Gemma in 4-bit (QLoRA)

In [ ]:
model_id = "google/gemma-4-e4b-it"  # Gemma 4 — fits Kaggle free tier (T4) with 4-bit QLoRA

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

## 4. Setup LoRA and Training Arguments

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = TrainingArguments(
    output_dir="./gemma-triage-results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=10,
    logging_steps=10,
    learning_rate=2e-4,
    max_steps=50,
    fp16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
)

## 5. Train the Model

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()

## 6. Save and Export for Ollama
After training, save the adapter. To use this in Ollama locally (like in the SANA backend):
1. Merge the LoRA weights with the base model.
2. Convert to GGUF using `llama.cpp`.
3. Create a `Modelfile` on your local machine: `FROM ./triage-gemma.gguf`.
4. Run `ollama create gemma4-triage -f Modelfile`.

In [ ]:
trainer.model.save_pretrained("gemma-triage-adapter")
tokenizer.save_pretrained("gemma-triage-adapter")
print("Training complete! Adapter saved.")